In [39]:
import numpy as np
import pandas as pd 
import matplotlib.pyplot as pyplot
import seaborn as sns
import sklearn
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score
from sklearn.feature_extraction import DictVectorizer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import roc_curve
from sklearn.metrics import auc

In [25]:
#!wget https://raw.githubusercontent.com/DataTalksClub/machine-learning-zoomcamp/main/cohorts/2026/data/course_lead_scoring_2026.csv

In [26]:
df = pd.read_csv("course_lead_scoring_2026.csv")

In [27]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 5000 entries, 0 to 4999
Data columns (total 9 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   lead_source               4852 non-null   str    
 1   industry                  4760 non-null   str    
 2   employment_status         4806 non-null   str    
 3   location                  4792 non-null   str    
 4   annual_income             4631 non-null   float64
 5   number_of_courses_viewed  5000 non-null   int64  
 6   interaction_count         5000 non-null   int64  
 7   lead_score                4965 non-null   float64
 8   converted                 5000 non-null   int64  
dtypes: float64(2), int64(3), str(4)
memory usage: 351.7 KB


In [28]:
#Data preparation
categorical_features = list((df.dtypes[df.dtypes == 'str']).index)
numerical_features = list((df.dtypes[df.dtypes != 'str']).index)

In [29]:
"""
If there are missing values:
For categorical features, replace them with 'NA'
For numerical features, replace with with 0.0 
"""
df[categorical_features] = df[categorical_features].fillna("NA")

df[numerical_features] = df[numerical_features].fillna(0)

In [30]:
#checking whether my fillna() works
df.isna().sum()

lead_source                 0
industry                    0
employment_status           0
location                    0
annual_income               0
number_of_courses_viewed    0
interaction_count           0
lead_score                  0
converted                   0
dtype: int64

In [31]:
#Splitting the data into 3 parts, train, test, val
df_full_train, df_test = train_test_split(df, test_size=0.2, random_state=1)
df_train, df_val = train_test_split(
    df_full_train, test_size=0.25, random_state=1
)

In [32]:
del numerical_features[-1]

In [33]:
y = df_train.converted.values
x_train = df_train[numerical_features]

In [34]:
#Question 1: ROC AUC feature importance
for feature in numerical_features:
    auc = roc_auc_score(y,x_train[feature])
    if auc < 0.5:
        auc = roc_auc_score(y,-x_train[feature])
    print(f"{feature}:: {round(auc,3)}")
        

annual_income:: 0.608
number_of_courses_viewed:: 0.723
interaction_count:: 0.765
lead_score:: 0.788


In [35]:
y_train = df_train.converted.values
X_train = df_train[categorical_features+numerical_features].reset_index(drop=True)

y_val = df_val.converted.values
X_val = df_val[categorical_features+numerical_features].reset_index(drop=True)

In [36]:
X_train

,lead_source,industry,employment_status,location,annual_income,number_of_courses_viewed,interaction_count,lead_score
0,paid_ads,technology,employed,asia,80198.0,5,9,0.91
1,paid_ads,retail,employed,asia,50548.0,2,5,0.48
2,NA,finance,employed,NA,76300.0,0,3,0.42
3,paid_ads,finance,employed,NA,80371.0,2,6,0.49
4,organic_search,finance,employed,asia,73567.0,2,4,0.48
...,...,...,...,...,...,...,...,...
2995,paid_ads,finance,employed,north_america,84573.0,2,3,0.37
2996,events,technology,employed,asia,85051.0,1,3,0.56
2997,events,retail,unemployed,asia,21561.0,0,1,0.17
2998,organic_search,healthcare,unemployed,south_america,31691.0,3,5,0.49


In [37]:
#Question 2: Training the model

#convert train and val data to dicts for one-hot transformers
train_dicts = X_train.to_dict(orient="records")
val_dicts = X_val.to_dict(orient="records")

#one-hot encoding to categorical variables
dv = DictVectorizer(sparse=False)
train_dicts = dv.fit_transform(train_dicts)
val_dicts = dv.transform(val_dicts)

#building and training the model
model = LogisticRegression(solver='liblinear', C=1.0, max_iter=1000)
model.fit(train_dicts,y_train)

#predicting the probability of being converted on val dataset
pred_prob = model.predict_proba(val_dicts)[:,1]

#calculating auc score
auc_score = roc_auc_score(y_val,pred_prob)
print(round(auc_score,3))

0.732


In [40]:
tpr,fpr,threshold = roc_curve(y_val,pred_prob)
round(auc(tpr,fpr),3)

0.732